# Hierarchical Supervisory Workflow (Magentic Orchestration)

This notebook demonstrates the **Hierarchical Supervisory pattern** using the Microsoft Agent Framework's `MagenticBuilder`. This pattern is also known as "Magentic" orchestration - a powerful workflow where a manager/orchestrator agent coordinates multiple specialized sub-agents to complete complex tasks.

## What is Magentic Orchestration?

Magentic orchestration is a hierarchical workflow pattern where:
- A **Manager Agent** (Supervisor) coordinates the overall task
- **Specialized Sub-Agents** handle specific aspects of the problem
- The manager decides which sub-agent to invoke based on the user's request
- Sub-agents can be called multiple times as needed to complete the task

## What You'll Learn

- How to create a **Manager Agent** that orchestrates workflow
- How to build **Specialized Sub-Agents** with specific tools and instructions
- How to use **MagenticBuilder** to wire everything together
- How the supervisor pattern enables complex, multi-step tasks

---

**Reference**: 
- [Microsoft Learn - Magentic Orchestration](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/magentic?pivots=programming-language-python)
- [GitHub Sample - Magentic](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/magentic.py)

## 1. Setup and Installations

First, let's install the required dependencies and set up the environment.

In [ ]:
# Install required dependencies
!pip install python-dotenv

In [1]:
# Import and setup environment
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework.openai import OpenAIChatClient

# Load environment variables
load_dotenv(find_dotenv())
import nest_asyncio
nest_asyncio.apply()
# Verify API keys are set
print(f"GROQ Endpoint: {os.environ.get('GROQ_ENDPOINT', 'Not set')}")
print(f"GROQ API Key: {'Set' if os.environ.get('GROQ_API_KEY') else 'Not set'}")

GROQ Endpoint: https://api.groq.com/openai/v1/
GROQ API Key: Set


In [2]:
# Create OPEN AI Chat Client for OpenRouter Models
# Using Groq as an alternative (faster, more reliable than Ollama for streaming)
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("GROQ_ENDPOINT"),
    api_key=os.environ.get("GROQ_API_KEY"),
    model_id="openai/gpt-oss-120b"
)

In [3]:
# from agent_framework.openai import OpenAIChatClient
# # Create OPEN AI Chat Client for OpenRouter Models
# openai_chat_client = OpenAIChatClient(
#     base_url=os.environ.get("OPENROUTER_ENDPOINT"),
#     api_key=os.environ.get("OPENROUTER_API_KEY"),
#     model_id="nvidia/nemotron-3-nano-30b-a3b:free"
# )

---

## 2. Understanding the Hierarchical Supervisory Pattern

### What is Magentic Orchestration?

The **Magentic** (or Hierarchical Supervisory) pattern is a workflow architecture where:

- A **Manager/Supervisor Agent** acts as the orchestrator
- The manager analyzes user requests and determines which specialized agent should handle them
- **Sub-Agents** each specialize in specific tasks or domains
- The manager coordinates and aggregates results from sub-agents

### When to Use This Pattern

- Complex tasks requiring multiple areas of expertise
- Tasks that can be broken into specialized sub-tasks
- When you want a single entry point (the manager) but diverse capabilities

### Key Components

- **Manager Agent**: The supervisor that coordinates the workflow
- **Participants**: The specialized sub-agents available to the manager
- **MagenticBuilder**: The builder class that constructs the workflow
- **Max Round Count**: Limits how many times the workflow can iterate

## 3. Import Required Libraries

Let's import the necessary components from the Microsoft Agent Framework, including the MagenticBuilder for creating our hierarchical supervisory workflow.

In [4]:
import asyncio
import os
import json
import requests
from random import randint
from typing import Annotated, Dict, Any, Optional

from agent_framework import tool, Agent
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential
from pydantic import Field

In [5]:
@tool
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."

@tool
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."

@tool
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."

In [6]:
import asyncio
import json
import requests
from typing import Annotated, cast

from agent_framework import (
    Message,
    WorkflowEvent,
    WorkflowRunState,
    tool,
)
from agent_framework._types import AgentResponseUpdate
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder

---

## 4. Create Food-Related Tools

We'll create tools that our specialized agents will use. These tools interact with TheMealDB API.

In [7]:
# Helper function to clean meal data from API
def _clean_meal_data(meal: Dict[str, Any]) -> Dict[str, Any]:
    """
    Helper function to restructure the raw meal API response into a clean, 
    LLM-friendly format by combining ingredients and measures.
    """
    if not meal:
        return {}

    # Combine ingredients and measures into a single list
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f"strIngredient{i}")
        measure = meal.get(f"strMeasure{i}")
        if ing and ing.strip():
            ingredients.append(f"{measure.strip()} {ing.strip()}".strip())

    return {
        "id": meal.get("idMeal"),
        "name": meal.get("strMeal"),
        "category": meal.get("strCategory"),
        "area": meal.get("strArea"),
        "instructions": meal.get("strInstructions"),
        "ingredients": ingredients,
        "tags": meal.get("strTags"),
        "youtube_link": meal.get("strYoutube"),
        "image_url": meal.get("strMealThumb")
    }

In [8]:
# Tool: Get a random meal
@tool
def get_random_meal() -> str:
    """
    Retrieves a random meal recipe from the database. 
    Use this when the user wants a surprise suggestion or explicitly asks for a random recommendation.

    Returns:
        str: A JSON string containing the meal name, ingredients, and cooking instructions.
    """
    try:
        response = requests.get("https://www.themealdb.com/api/json/v1/1/random.php")
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": "No meal found."})
            
        meal = _clean_meal_data(data["meals"][0])
        return json.dumps(meal, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch random meal: {str(e)}"})

In [9]:
# Tool: Get meal by name
@tool
def get_meal_by_name(meal_name: Annotated[str, "The name of the meal to search for"]) -> str:
    """
    Search for a specific meal by its name.
    Use this when the user specifies a particular dish they want to make or learn about.

    Args:
        meal_name (str): The name of the meal to search for.

    Returns:
        str: A JSON string containing the meal details or search results.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/search.php?s={meal_name}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meal found with name '{meal_name}'"})
        
        meals = [_clean_meal_data(meal) for meal in data["meals"]]
        return json.dumps(meals, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch meal: {str(e)}"})

In [10]:
# Tool: Get meals by category
@tool
def get_meals_by_category(category: Annotated[str, "The category name (e.g., Beef, Chicken, Dessert)"]) -> str:
    """
    Get a list of meals from a specific category (e.g., Beef, Chicken, Dessert).
    Use this when the user wants to explore meals from a specific category.

    Args:
        category (str): The category name (e.g., "Beef", "Chicken", "Dessert").

    Returns:
        str: A JSON string containing list of meals in that category.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/filter.php?c={category}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meals found in category '{category}'"})
        
        meals = [{"id": m["idMeal"], "name": m["strMeal"], "thumbnail": m["strMealThumb"]} 
                 for m in data["meals"]]
        return json.dumps(meals[:10], indent=2)  # Limit to 10 results
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch category: {str(e)}"})

In [11]:
# Tool: Get meals by area/cuisine
@tool
def get_meals_by_area(area: Annotated[str, "The area/cuisine name (e.g., Canadian, Chinese, Italian)"]) -> str:
    """
    Get a list of meals from a specific area/cuisine (e.g., Canadian, Chinese, Italian).
    Use this when the user wants to explore food from a specific cuisine/region.

    Args:
        area (str): The area/cuisine name (e.g., "Canadian", "Chinese", "Italian").

    Returns:
        str: A JSON string containing list of meals from that area.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/filter.php?a={area}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meals found for area '{area}'"})
        
        meals = [{"id": m["idMeal"], "name": m["strMeal"], "thumbnail": m["strMealThumb"]} 
                 for m in data["meals"]]
        return json.dumps(meals[:10], indent=2)  # Limit to 10 results
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch area: {str(e)}"})

In [12]:
# Tool: Get all categories
@tool
def get_all_categories() -> str:
    """
    Get a list of all available meal categories.
    Use this to show the user what categories are available.

    Returns:
        str: A JSON string containing all available categories.
    """
    try:
        response = requests.get("https://www.themealdb.com/api/json/v1/1/categories.php")
        response.raise_for_status()
        data = response.json()
        
        if not data.get("categories"):
            return json.dumps({"error": "No categories found"})
        
        categories = [c["strCategory"] for c in data["categories"]]
        return json.dumps({"categories": categories}, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch categories: {str(e)}"})

---

## 5. Create the Chat Client

Now let's create the OpenAI chat client that will be used by all our agents.

---

## 6. Create Specialized Sub-Agents (Participants)

In the Magentic pattern, these sub-agents are called **participants**. They are the specialized workers that the manager can invoke. Each agent has a specific focus:

- **Meal Details Agent**: Provides detailed information about specific meals (by name)
- **Category Explorer Agent**: Helps users explore meals by category (Beef, Chicken, etc.)
- **Cuisine Explorer Agent**: Helps users explore meals by cuisine/area (Italian, Chinese, etc.)
- **Random Surprise Agent**: Suggests random meals for indecisive users

These participants will be registered with the MagenticBuilder to be available for the manager agent to call upon.

In [13]:
# Sub-Agent 1: Meal Details Agent
# This agent specializes in providing detailed information about specific meals

meal_details_agent = Agent(
    name="meal_details_agent",
    client=openai_chat_client,
    instructions="""You are a meal details expert. Your role is to provide comprehensive information about specific meals.
    
    When a user asks about a specific meal:
    1. Use the get_meal_by_name tool to find the meal
    2. Present the information in a friendly, organized manner
    3. Include: name, category, cuisine area, ingredients list, and cooking instructions
    4. If available, mention any YouTube tutorial links
    
    Always be helpful and thorough in your explanations.""",
    tools=[get_meal_by_name]
)

In [14]:
# Sub-Agent 2: Category Explorer Agent
# This agent helps users explore meals by category

category_explorer_agent = Agent(
    name="category_explorer_agent",
    client=openai_chat_client,
    instructions="""You are a category exploration expert. Your role is to help users discover meals by category.
    
    When a user wants to explore meals by category:
    1. First use get_all_categories to show available categories
    2. Ask the user which category interests them
    3. Use get_meals_by_category to list meals in that category
    4. Present the results in a clear, organized list
    
    Be enthusiastic and help guide users to discover new food categories!""",
    tools=[get_all_categories, get_meals_by_category]
)

In [15]:
# Sub-Agent 3: Cuisine Explorer Agent
# This agent helps users explore meals by cuisine/area

cuisine_explorer_agent = Agent(
    name="cuisine_explorer_agent",
    client=openai_chat_client,
    instructions="""You are a cuisine exploration expert. Your role is to help users discover meals from different world cuisines.
    
    When a user wants to explore meals by cuisine:
    1. Show popular cuisine areas (e.g., Italian, Chinese, Mexican, Indian, Japanese, etc.)
    2. Ask the user which cuisine they are interested in
    3. Use get_meals_by_area to list meals from that cuisine
    4. Present the results highlighting the variety of dishes available
    
    Be passionate about world cuisines and help users explore new flavors!""",
    tools=[get_meals_by_area]
)

In [16]:
# Sub-Agent 4: Random Surprise Agent
# This agent suggests random meals for users who cannot decide

random_surprise_agent = Agent(
    name="random_surprise_agent",
    client=openai_chat_client,
    instructions="""You are a food surprise expert. Your role is to help indecisive users by suggesting random meals.
    
    When a user cannot decide what to eat or wants a surprise:
    1. Use get_random_meal to get a random meal suggestion
    2. Present the meal in an exciting, appetizing way
    3. Highlight what makes this dish special
    4. Ask if they would like another suggestion or more details
    
    Be enthusiastic and make the surprise feel exciting!""",
    tools=[get_random_meal]
)

In [17]:
# Create the Manager Agent (Supervisor)
# This agent will coordinate the workflow by deciding which participant to invoke

manager_agent = Agent(
    name="MagenticFoodManager",
    description="Orchestrator that coordinates the Food Workflow by efficiently analyzing users query",
    instructions="You coordinate a team to complete complex Food based tasks efficiently.",
    client=openai_chat_client
)

In [18]:
# Build the Magentic Workflow (Hierarchical Supervisor)
# MagenticBuilder creates a workflow where the manager agent coordinates participant agents

from agent_framework.orchestrations import MagenticBuilder

workflow = MagenticBuilder(
    participants=[random_surprise_agent, meal_details_agent, category_explorer_agent, cuisine_explorer_agent],
    intermediate_outputs=True,        # Enable intermediate output for debugging
    manager_agent=manager_agent,      # The supervisor agent
    max_round_count=10,               # Maximum workflow iterations
    max_stall_count=3,                # Stop after 3 rounds with no progress
    max_reset_count=2                 # Allow 2 workflow resets
).build()

In [19]:
# Run the Magentic Workflow
# The workflow processes the user's request by having the manager coordinate participant agents

import json
import asyncio
from typing import cast

from agent_framework import (
    AgentResponseUpdate,
    Message,
    WorkflowEvent,
)
from agent_framework.orchestrations import MagenticProgressLedger

# Sample task that requires the manager to coordinate multiple agents
task = (
    "I want to eat a healthy meal daily with 10g protein as recommended by dietician. "
    "Suggest me meal to have at lunch time so that I have enough energy for the day. "
    "Make sure to include some sides which complement the taste"
)

# Keep track of the last executor to format output nicely in streaming mode
last_message_id: str | None = None
output_event: WorkflowEvent | None = None
async for event in workflow.run(task, stream=True):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        message_id = event.data.message_id
        if message_id != last_message_id:
            if last_message_id is not None:
                print("\n")
            print(f"- {event.executor_id}:", end=" ", flush=True)
            last_message_id = message_id
        print(event.data, end="", flush=True)

    elif event.type == "magentic_orchestrator":
        print(f"\n[Magentic Orchestrator Event] Type: {event.data.event_type.name}")
        if isinstance(event.data.content, MagenticProgressLedger):
            print(f"Please review progress ledger:\n{json.dumps(event.data.content.to_dict(), indent=2)}")
        else:
            print(f"Unknown data type in MagenticOrchestratorEvent: {type(event.data.content)}")

        # Block to allow user to read the plan/progress before continuing
        # Note: this is for demonstration only and is not the recommended way to handle human interaction.
        # Please refer to `with_plan_review` for proper human interaction during planning phases.
        await asyncio.get_event_loop().run_in_executor(None, input, "Press Enter to continue...")

    elif event.type == "output":
        output_event = event

# The output of the Magentic workflow is a list of ChatMessages with only one final message
# generated by the orchestrator.
output_messages = cast(list[Message], output_event.data)
output = output_messages[-1].text
print(output)


[Magentic Orchestrator Event] Type: PLAN_CREATED
Unknown data type in MagenticOrchestratorEvent: <class 'agent_framework._types.Message'>


Press Enter to continue... 



[Magentic Orchestrator Event] Type: PROGRESS_LEDGER_UPDATED
Please review progress ledger:
{
  "is_request_satisfied": {
    "reason": "The user has not yet received a concrete lunch recommendation with protein content and complementary sides; only a planning outline was provided.",
    "answer": false
  },
  "is_in_loop": {
    "reason": "We have moved from the initial user request to a planning stage and have not repeated any previous actions; no loop is detected.",
    "answer": false
  },
  "is_progress_being_made": {
    "reason": "The assistant has outlined a clear plan and identified the next step (detail the meal), showing forward progress toward fulfilling the request.",
    "answer": true
  },
  "next_speaker": {
    "reason": "We need concrete nutritional details and a ready\u2011to\u2011use lunch menu that meets the 10\u202fg protein target and includes tasty sides.",
    "answer": "meal_details_agent"
  },
  "instruction_or_question": {
    "reason": "Provide the actual l

Press Enter to continue... 


- meal_details_agent: ### Balanced Lunch‑Menu (≈10 g protein)

| Component | Portion | Protein | Calories | Quick Nutrient Highlights |
|-----------|---------|---------|----------|---------------------------|
| **Main** | **Quinoa‑and‑Black‑Bean Power Bowl** – ½ cup cooked quinoa + ¼ cup cooked black beans (mixed with a squeeze of lime, chopped cilantro, and a pinch of cumin) | **≈ 9 g** (Quinoa ≈ 4 g, Black beans ≈ 5 g) | **≈ 210 kcal** | • Complex carbs & fiber from quinoa<br>• Plant‑based protein & iron from beans<br>• Magnesium & potassium |
| **Side 1** | **Baby‑spinach & cherry‑tomato salad** – 1 cup baby spinach, ½ cup halved cherry tomatoes, 1 tbsp sliced almonds | **≈ 2 g** (mostly from almonds) | **≈ 70 kcal** | • Vitamin K, A, C from spinach & tomato<br>• Healthy monounsaturated fats & vitamin E from almonds |
| **Side 2** *(optional but recommended for healthy fats & satiety)* | **Roasted avocado wedges** – ¼ medium avocado, tossed with a dash of smoked paprika and a drizzl

Press Enter to continue... 


## Lunch‑Ready Meal that Gives ≈ 10 g Protein  
*(All portions are for one serving. Adjust the quantities slightly if you need the protein to be exactly 10 g.)*

| Component | Portion & How to Prepare | Protein | Calories | Why It Works |
|-----------|--------------------------|---------|----------|--------------|
| **Main** | **Quinoa‑Black‑Bean Bowl** – ½ cup cooked quinoa mixed with ¼ cup cooked black beans, seasoned with a squeeze of lime, ¼ tsp cumin, and a handful of chopped fresh cilantro. | **≈ 9 g** (quinoa ≈ 4 g, black beans ≈ 5 g) | **≈ 210 kcal** | • Quinoa is a complete grain (contains all essential amino acids). <br>• Black beans add extra protein, fiber, iron and keep blood‑sugar steady. |
| **Side 1** | **Baby‑Spinach & Cherry‑Tomato Salad** – 1 cup baby spinach, ½ cup halved cherry tomatoes, 1 tbsp sliced almonds, tossed with a drizzle of ½ tsp olive oil and a splash of balsamic vinegar. | **≈ 2 g** (mostly from almonds) | **≈ 70 kcal** | • Spinach supplies vitamin K, 

In [ ]:
#---

## 7. Run the Magentic Workflow

# Now let's execute our hierarchical supervisory workflow. The manager agent will analyze the user's request and coordinate the appropriate participant agents to complete the task.

In [20]:
from agent_framework import WorkflowViz

In [22]:
!pip install graphviz

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)


In [23]:
viz = WorkflowViz(workflow)
print(viz.save_png("simple_concurrent_workflow.png"))

simple_concurrent_workflow.png
